Layers: PW supply · Clusters · TDS · Demand · SWD wells · Oil wells

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import folium
from folium.plugins import MarkerCluster
import os

In [ ]:
N_COLAB = 'google.colab' in str(dir())
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK_DIR = "/content/drive/MyDrive/YOUR_FOLDER/pw_analysis"
else:
    WORK_DIR = "."

DATA_DIR = f"{WORK_DIR}/data_raw/NM geofiles"
OUT_DIR = f"{WORK_DIR}/outputs"

LOAD ALL DATA

In [ ]:
# 1a. County boundaries
nm_counties = gpd.read_file(f"{DATA_DIR}/nm_counties.geojson")
nm_counties["county_key"] = nm_counties["NAME"].str.strip().str.lower()
print(f"✓ Counties: {len(nm_counties)}")

In [ ]:
# 1b. PW supply + cross-validation (county level)
supply = pd.read_csv(f"{PROC_DIR}/data_processed/nm_county_summary.csv")
supply["county_key"] = supply["county_key"].str.strip().str.lower()
print(f"✓ Supply data: {len(supply)} counties")

In [ ]:
# 1c. Demand (county level)
demand = pd.read_csv(f"{WORK_DIR}/data_processed/nm_water_demand_2025_final.csv")
demand["county_key"] = (demand["county_key"]
    .str.strip()
    .str.lower()
    .str.replace("_", " ")
)
print(f"✓ Demand data: {len(demand)} counties")

In [ ]:
# 1d. PW clusters — for TDS (township level, aggregated to county)
#     Adjust filename if different in your Drive
nm_pw = pd.read_csv(f"{PROC_DIR}/data_processed/nm_cross_validation.csv")
nm_pw["county_key"] = nm_pw["county"].str.strip().str.lower()
print(f"✓ PW clusters: {len(nm_pw)} townships")

In [ ]:
# 1e. Well point data
swd_wells = gpd.read_file(f"{PROC_DIR}/data_processed/ocd/nm_ocd_disposal_wells.geojson")
oil_wells = gpd.read_file(f"{PROC_DIR}/data_processed/ocd/nm_ocd_oil_wells.geojson")
oil_sample = oil_wells.sample(n=min(5000, len(oil_wells)), random_state=42)
print(f"✓ SWD wells: {len(swd_wells):,}")
print(f"✓ Oil sample: {len(oil_sample):,}")

COUNTY-LEVEL GEODATAFRAME

In [ ]:
#2a. TDS — aggregate from township to county
county_tds = (nm_pw
    .groupby("county_key")
    .agg(
        mean_tds_mgl   = ("tds_real", "mean"),
        median_tds_mgl = ("tds_real", "median"),
        min_tds_mgl    = ("tds_real", "min"),
        max_tds_mgl    = ("tds_real", "max"),
    )
    .round(1)
    .reset_index()
)

In [ ]:
# 2b. Merge everything onto county polygons
nm_map = (nm_counties
    .merge(supply,     on="county_key", how="left")
    .merge(demand[["county_key","agri_tpw_bbl_2025",
                   "power_tw_bbl_2025","total_demand_bbl_2025",
                   "plant_count"]],
           on="county_key", how="left")
    .merge(county_tds, on="county_key", how="left")
)

In [ ]:
# Fill NaN
nm_map["total_pw_vol"]          = nm_map["total_pw_vol"].fillna(0)
nm_map["dominant_cluster"]      = nm_map["dominant_cluster"].fillna(-1)
nm_map["swd_well_count"]        = nm_map["swd_well_count"].fillna(0).astype(int)
nm_map["total_demand_bbl_2025"] = nm_map["total_demand_bbl_2025"].fillna(0)
nm_map["agri_tpw_bbl_2025"]    = nm_map["agri_tpw_bbl_2025"].fillna(0)
nm_map["power_tw_bbl_2025"]    = nm_map["power_tw_bbl_2025"].fillna(0)
nm_map["plant_count"]          = nm_map["plant_count"].fillna(0).astype(int)
nm_map["mean_tds_mgl"]         = nm_map["mean_tds_mgl"].fillna(0)

In [ ]:
print(f"\n✓ Master GeoDataFrame: {len(nm_map)} counties")
print(f"  Counties with PW data:     {(nm_map['total_pw_vol']>0).sum()}")
print(f"  Counties with demand data: {(nm_map['total_demand_bbl_2025']>0).sum()}")
print(f"  Counties with TDS data:    {(nm_map['mean_tds_mgl']>0).sum()}")
print(f"\nColumns: {nm_map.columns.tolist()}")

THE MAP

In [ ]:
def fmt_bbl(val):
    if pd.isna(val) or val == 0: return "No data"
    if val >= 1e9:  return f"{val/1e9:.2f}B bbl"
    if val >= 1e6:  return f"{val/1e6:.1f}M bbl"
    if val >= 1e3:  return f"{val/1e3:.1f}K bbl"
    return f"{val:.0f} bbl"

# Add formatted columns to nm_map
nm_map["annual_supply_bbl"] = nm_map["total_pw_vol"] / 5
nm_map["annual_supply_fmt"] = nm_map["annual_supply_bbl"].apply(
    lambda x: fmt_bbl(x) + "/yr" if x > 0 else "No data")
nm_map["pw_vol_fmt"]        = nm_map["total_pw_vol"].apply(
    lambda x: fmt_bbl(x) + " (5yr)")
nm_map["demand_fmt"]        = nm_map["total_demand_bbl_2025"].apply(
    lambda x: fmt_bbl(x) + "/yr" if x > 0 else "No data")
nm_map["agri_fmt"]          = nm_map["agri_tpw_bbl_2025"].apply(
    lambda x: fmt_bbl(x) + "/yr" if x > 0 else "No data")
nm_map["power_fmt"]         = nm_map["power_tw_bbl_2025"].apply(
    lambda x: fmt_bbl(x) + "/yr" if x > 0 else "No data")
nm_map["tds_fmt"]           = nm_map["mean_tds_mgl"].apply(
    lambda x: f"{x:,.0f} mg/L" if x > 0 else "No data")
nm_map["ratio_fmt"]         = (
    nm_map["annual_supply_bbl"] /
    nm_map["total_demand_bbl_2025"].replace(0, float("nan"))
).round(2).apply(
    lambda x: f"{x:.2f}x supply vs demand" if pd.notna(x) else "No demand data")

# Confirm columns exist
print("✓ Formatted columns added:")
print(nm_map[["NAME","annual_supply_fmt","demand_fmt","ratio_fmt"]]
      .dropna(subset=["ratio_fmt"]).to_string(index=False))

In [ ]:
CLUSTER_COLORS = {
    -1: "#F1EFE8",   # no data
     0: "#93C5FD",   # high volume
     1: "#34D399",   # moderate
     2: "#FBBF24",   # low-medium
     3: "#EF4444",   # very low
}

m = folium.Map(location=[34.5, -106.0], zoom_start=7,
               tiles="CartoDB positron")

In [ ]:
# ── LAYER 1: PW volume choropleth ──────────────────────────────────
folium.Choropleth(
    geo_data     = nm_map.to_json(),
    data         = nm_map,
    columns      = ["county_key", "total_pw_vol"],
    key_on       = "feature.properties.county_key",
    fill_color   = "Blues",
    fill_opacity = 0.65,
    line_opacity = 0.3,
    nan_fill_color   = "#F1EFE8",
    nan_fill_opacity = 0.3,
    legend_name  = "PW volume — 5yr (bbl)",
    name         = "PW volume (supply)",
    show         = True
).add_to(m)

In [ ]:
# ── LAYER 2: PW cluster overlay ────────────────────────────────────
folium.GeoJson(
    nm_map,
    name="PW clusters",
    style_function=lambda f: {
        "fillColor": CLUSTER_COLORS.get(
            int(f["properties"].get("dominant_cluster", -1)), "#F1EFE8"),
        "color": "white", "weight": 0.8, "fillOpacity": 0.65
    },
    show=False
).add_to(m)

In [ ]:
# ── LAYER 3: TDS choropleth ────────────────────────────────────────
folium.Choropleth(
    geo_data     = nm_map.to_json(),
    data         = nm_map,
    columns      = ["county_key", "mean_tds_mgl"],
    key_on       = "feature.properties.county_key",
    fill_color   = "RdYlGn_r",  # green=low TDS (easy treat), red=high
    fill_opacity = 0.70,
    line_opacity = 0.2,
    nan_fill_color   = "#F1EFE8",
    nan_fill_opacity = 0.3,
    legend_name  = "Mean TDS — county avg (mg/L)",
    name         = "TDS salinity (mg/L)",
    show         = False
).add_to(m)

In [ ]:
# ── LAYER 4: Total water demand 2025 ───────────────────────────────
folium.Choropleth(
    geo_data     = nm_map.to_json(),
    data         = nm_map,
    columns      = ["county_key", "total_demand_bbl_2025"],
    key_on       = "feature.properties.county_key",
    fill_color   = "YlOrRd",
    fill_opacity = 0.65,
    line_opacity = 0.2,
    nan_fill_color   = "#F1EFE8",
    nan_fill_opacity = 0.3,
    legend_name  = "Total water demand 2025 — agri + power (bbl/yr)",
    name         = "Total demand 2025",
    show         = False
).add_to(m)

In [ ]:
# ── LAYER 5: Agriculture demand only ───────────────────────────────
folium.Choropleth(
    geo_data     = nm_map.to_json(),
    data         = nm_map,
    columns      = ["county_key", "agri_tpw_bbl_2025"],
    key_on       = "feature.properties.county_key",
    fill_color   = "Greens",
    fill_opacity = 0.65,
    line_opacity = 0.2,
    nan_fill_color   = "#F1EFE8",
    nan_fill_opacity = 0.3,
    legend_name  = "Irrigation demand 2025 (bbl/yr)",
    name         = "Irrigation demand 2025",
    show         = False
).add_to(m)

In [ ]:
# ── LAYER 6: Power demand only ─────────────────────────────────────
folium.Choropleth(
    geo_data     = nm_map.to_json(),
    data         = nm_map,
    columns      = ["county_key", "power_tw_bbl_2025"],
    key_on       = "feature.properties.county_key",
    fill_color   = "Oranges",
    fill_opacity = 0.65,
    line_opacity = 0.2,
    nan_fill_color   = "#F1EFE8",
    nan_fill_opacity = 0.3,
    legend_name  = "Power plant demand 2025 (bbl/yr)",
    name         = "Power demand 2025",
    show         = False
).add_to(m)


In [ ]:
folium.GeoJson(
    nm_map,
    name="County labels",
    style_function=lambda x: {"fillOpacity": 0, "weight": 0},
    tooltip=folium.GeoJsonTooltip(
        fields=["NAME","annual_supply_fmt","pw_vol_fmt",
                "dominant_cluster","swd_well_count",
                "tds_fmt","demand_fmt","agri_fmt",
                "power_fmt","ratio_fmt"],
        aliases=["County:","PW supply (annual):",
                 "PW vol (5yr total):","Cluster:",
                 "SWD wells:","Mean TDS:",
                 "Total demand 2025:","Irrigation 2025:",
                 "Power 2025:","Supply vs demand:"],
        localize=False
    ),
    show=True
).add_to(m)
print("✓ County layers added")

In [ ]:
# ── LAYER 8: SWD wells ─────────────────────────────────────────────
swd_group = folium.FeatureGroup(name="SWD wells", show=True)
for _, row in swd_wells.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=4, color="#059669", fill=True,
        fill_color="#1D9E75", fill_opacity=0.8, weight=0.5,
        popup=folium.Popup(
            f"<b>{row.get('name','SWD Well')}</b><br>"
            f"County: {str(row.get('county','')).title()}<br>"
            f"Status: {row.get('status','')}",
            max_width=200)
    ).add_to(swd_group)
swd_group.add_to(m)

In [ ]:
# ── LAYER 9: Oil wells (cross-validation, sampled) ─────────────────
oil_cluster = MarkerCluster(
    name="Oil wells (5k sample — cross-validation)",
    show=False)
for _, row in oil_sample.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=2, color="#64748B", fill=True,
        fill_color="#888780", fill_opacity=0.5, weight=0,
        popup=folium.Popup(
            f"{row.get('name','Oil well')}<br>"
            f"County: {str(row.get('county','')).title()}",
            max_width=150)
    ).add_to(oil_cluster)
oil_cluster.add_to(m)

In [ ]:
print(f"✓ SWD layer: {len(swd_wells):,} wells")
print(f"✓ Oil layer: {len(oil_sample):,} sampled wells")

In [ ]:
# ── Legend ──────────────────────────────────────────────────────────
legend_html = """
<div style="position:fixed;bottom:30px;right:10px;z-index:1000;
  background:white;padding:14px 18px;border-radius:10px;
  border:1px solid #CBD5E1;font-family:sans-serif;font-size:11px;
  box-shadow:0 2px 10px rgba(0,0,0,0.12);min-width:240px">
  <b style="font-size:13px">NM Produced Water Map</b><br>
  <span style="color:#64748B;font-size:10px">
    Cross-validation of independent data</span><br><br>

  <b style="color:#475569;font-size:10px">COUNTY LAYERS (toggle)</b><br>
  <span style="background:#2563EB;display:inline-block;width:12px;height:12px;
    opacity:.65;border-radius:2px;vertical-align:middle"></span>
  &nbsp;PW volume — supply<br>
  <span style="background:#FBBF24;display:inline-block;width:12px;height:12px;
    opacity:.65;border-radius:2px;vertical-align:middle"></span>
  &nbsp;PW clusters<br>
  <span style="background:#D73027;display:inline-block;width:12px;height:12px;
    opacity:.65;border-radius:2px;vertical-align:middle"></span>
  &nbsp;Mean TDS (red=high salinity)<br>
  <span style="background:#FC8D59;display:inline-block;width:12px;height:12px;
    opacity:.65;border-radius:2px;vertical-align:middle"></span>
  &nbsp;Total water demand 2025<br>
  <span style="background:#74C476;display:inline-block;width:12px;height:12px;
    opacity:.65;border-radius:2px;vertical-align:middle"></span>
  &nbsp;Irrigation demand 2025<br><br>

  <b style="color:#475569;font-size:10px">TDS TREATMENT GUIDE</b><br>
  <span style="color:#1D9E75">&#9632;</span>
  &nbsp;&lt;10k mg/L — Brackish RO<br>
  <span style="color:#59A4C7">&#9632;</span>
  &nbsp;10–35k mg/L — Seawater RO<br>
  <span style="color:#F59E0B">&#9632;</span>
  &nbsp;35–100k mg/L — High-P RO<br>
  <span style="color:#DC2626">&#9632;</span>
  &nbsp;&gt;100k mg/L — Thermal only<br><br>

  <b style="color:#475569;font-size:10px">WELL INFRASTRUCTURE</b><br>
  <span style="background:#1D9E75;display:inline-block;width:9px;height:9px;
    border-radius:50%;vertical-align:middle"></span>
  &nbsp;SWD wells<br>
  <span style="background:#888780;display:inline-block;width:9px;height:9px;
    border-radius:50%;vertical-align:middle"></span>
  &nbsp;Oil wells (sample)
</div>"""

m.get_root().html.add_child(folium.Element(legend_html))
folium.LayerControl(collapsed=True).add_to(m)

In [ ]:
OUTPUT_FILE = f"{OUT_DIR}/nm_full_map.html"
m.save(OUTPUT_FILE)
print(f"\n✓ Saved: nm_full_map.html")